# 02 | Data correction: road network, landslide susceptibility, barangays

**Stage:** Data correction (Stage 2)  
**Scope of this notebook:** investigation of road-network issues flagged in `01_validate_datasets.ipynb`. Susceptibility and barangay sections will be added after this.  
**Rule:** nothing in `data/raw/` is edited. Investigation outputs go to `data/processed/correction/`; decisions go to `docs/data_correction_log.csv`.

## 1. Configuration

In [1]:
from pathlib import Path
import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
from shapely import wkt
from shapely.geometry import LineString, Point
from IPython.display import display

PROJECTED_CRS = 'EPSG:32651'
GEOGRAPHIC_CRS = 'EPSG:4326'
LONG_STRAIGHT_EDGE_M = 100.0   # Review threshold for edges with no stored geometry.

ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / 'data' / 'raw').is_dir()), None
)
if ROOT is None:
    raise FileNotFoundError('Repository root not found: expected data/raw/.')
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'data' / 'processed' / 'correction'
OUT.mkdir(parents=True, exist_ok=True)
LOG_PATH = ROOT / 'docs' / 'data_correction_log.csv'

PATHS = {
    'barangays': RAW / 'Baguio_Barangaysv2.geojson',
    'roads': RAW / 'baguio_roads_full.graphml',
    'landslides': RAW / 'geoson_query.geojson',
}
print(f'Repository: {ROOT}')
print(f'Planar CRS: {PROJECTED_CRS}')

Repository: /Users/francie/Downloads/baguio-road-accessibility-ml
Planar CRS: EPSG:32651


## 2. Load the road graph and convert it to GeoDataFrames

GraphML stores every attribute as text, so coordinates and lengths are converted to numbers. Edges without a stored `geometry` are drawn as straight lines between their two nodes and flagged with `has_geometry = False`.

In [2]:
G = nx.read_graphml(PATHS['roads'], force_multigraph=True)
print(G)

nodes = gpd.GeoDataFrame(
    [{'node_id': n, 'street_count': d.get('street_count')} for n, d in G.nodes(data=True)],
    geometry=[Point(float(d['x']), float(d['y'])) for _, d in G.nodes(data=True)],
    crs=GEOGRAPHIC_CRS,
).set_index('node_id')

edge_rows, edge_geoms = [], []
for u, v, k, d in G.edges(keys=True, data=True):
    has_geom = 'geometry' in d
    geom = wkt.loads(d['geometry']) if has_geom else LineString(
        [nodes.geometry[u].coords[0], nodes.geometry[v].coords[0]])
    edge_rows.append({
        'u': u, 'v': v, 'key': k,
        'osmid': d.get('osmid'), 'name': d.get('name'), 'highway': d.get('highway'),
        'oneway': d.get('oneway'), 'length_m': float(d['length']),
        'has_geometry': has_geom,
    })
    edge_geoms.append(geom)
edges = gpd.GeoDataFrame(edge_rows, geometry=edge_geoms, crs=GEOGRAPHIC_CRS)
print(f'{len(nodes)} nodes, {len(edges)} edges')

MultiDiGraph with 3745 nodes and 8077 edges
3745 nodes, 8077 edges


## 3. Weakly connected components

Component 0 is the main network. Every other component is a fragment to investigate.

In [3]:
components = sorted(nx.weakly_connected_components(G), key=len, reverse=True)
node_component = {n: i for i, comp in enumerate(components) for n in comp}
nodes['component'] = nodes.index.map(node_component)
edges['component'] = edges['u'].map(node_component)

city = gpd.read_file(PATHS['barangays']).to_crs(PROJECTED_CRS).union_all()
nodes_proj = nodes.to_crs(PROJECTED_CRS)

rows = []
for i, comp in enumerate(components[1:], start=1):
    comp_nodes = nodes_proj[nodes_proj.component == i]
    comp_edges = edges[edges.component == i]
    centre = nodes.loc[list(comp)].union_all().centroid
    rows.append({
        'component': i,
        'n_nodes': len(comp),
        'n_edges': len(comp_edges),
        'total_length_m': round(comp_edges.length_m.sum()),
        'highway_types': ', '.join(sorted(set(map(str, comp_edges.highway.dropna())))) or '(no edges)',
        'road_names': ', '.join(sorted(set(comp_edges.name.dropna()))) or '(unnamed)',
        'dist_to_city_boundary_m': round(comp_nodes.distance(city.boundary).min()),
        'lat': round(centre.y, 6), 'lon': round(centre.x, 6),
        'osm_link': f'https://www.openstreetmap.org/#map=19/{centre.y:.6f}/{centre.x:.6f}',
    })
fragments = pd.DataFrame(rows)
pd.set_option('display.max_colwidth', 80)
display(fragments)

,component,n_nodes,n_edges,total_length_m,highway_types,road_names,dist_to_city_boundary_m,lat,lon,osm_link
0,1,18,36,4583,residential,"Almaciga, Cherry, Douglas Fir, Giant Sequoia, Mahogany, Mockernut, Narra, Sw...",2246,16.390363,120.605253,https://www.openstreetmap.org/#map=19/16.390363/120.605253
1,2,14,34,2171,"residential, unclassified",(unnamed),375,16.394737,120.561598,https://www.openstreetmap.org/#map=19/16.394737/120.561598
2,3,7,12,1147,residential,"Lias Road, Santo Tomas Central Road",416,16.373391,120.579472,https://www.openstreetmap.org/#map=19/16.373391/120.579472
3,4,7,12,2324,residential,Santo Tomas School Area Road,157,16.372047,120.585778,https://www.openstreetmap.org/#map=19/16.372047/120.585778
4,5,4,6,245,residential,(unnamed),127,16.379429,120.566346,https://www.openstreetmap.org/#map=19/16.379429/120.566346
5,6,2,2,410,residential,Woods Gate Square Alley 2,2453,16.389182,120.602779,https://www.openstreetmap.org/#map=19/16.389182/120.602779
6,7,2,2,106,residential,(unnamed),1522,16.381772,120.590733,https://www.openstreetmap.org/#map=19/16.381772/120.590733
7,8,2,2,781,residential,Kellet Street,2543,16.390825,120.603794,https://www.openstreetmap.org/#map=19/16.390825/120.603794
8,9,2,2,307,unclassified,(unnamed),209,16.379679,120.567000,https://www.openstreetmap.org/#map=19/16.379679/120.567000
9,10,2,2,178,residential,(unnamed),1485,16.387105,120.586544,https://www.openstreetmap.org/#map=19/16.387105/120.586544


## 4. Export for QGIS

Open `road_investigation.gpkg` in QGIS, add an OpenStreetMap basemap (XYZ Tiles → OpenStreetMap) and a satellite basemap, and inspect each fragment. Style the `fragments` layer by `component`.

In [4]:
gpkg = OUT / 'road_investigation.gpkg'
edges[edges.component == 0].to_file(gpkg, layer='main_network', driver='GPKG')
edges[edges.component > 0].to_file(gpkg, layer='fragments', driver='GPKG')
nodes[nodes.component > 0].reset_index().to_file(gpkg, layer='fragment_nodes', driver='GPKG')
fragments.to_csv(OUT / 'road_fragments.csv', index=False)
print(f'Saved {gpkg}')

Saved /Users/francie/Downloads/baguio-road-accessibility-ml/data/processed/correction/road_investigation.gpkg


## 5. Long edges with no stored geometry

In a simplified OSMnx graph, a missing `geometry` normally means the road is a straight line between two nodes, which is fine for short segments. Long ones are worth checking against satellite imagery, because few long roads in Baguio are straight.

In [5]:
straight = edges[~edges.has_geometry].copy()
print(f'{len(straight)} of {len(edges)} edges have no stored geometry')

long_straight = (straight[straight.length_m > LONG_STRAIGHT_EDGE_M]
                 .drop_duplicates(subset=['osmid', 'length_m'])   # one row per two-way pair
                 .sort_values('length_m', ascending=False))
c = long_straight.to_crs(PROJECTED_CRS).geometry.centroid.to_crs(GEOGRAPHIC_CRS)
long_straight['osm_link'] = [f'https://www.openstreetmap.org/#map=19/{p.y:.6f}/{p.x:.6f}' for p in c]
print(f'{len(long_straight)} unique edges longer than {LONG_STRAIGHT_EDGE_M:.0f} m to review')
display(long_straight[['u', 'v', 'key', 'name', 'highway', 'length_m', 'component', 'osm_link']].head(30))
long_straight.to_file(gpkg, layer='long_straight_edges', driver='GPKG')

1437 of 8077 edges have no stored geometry
17 unique edges longer than 100 m to review


,u,v,key,name,highway,length_m,component,osm_link
1658,529476141,529463768,0,Sunflower Street,residential,191.066614,0,https://www.openstreetmap.org/#map=19/16.417974/120.611332
3233,1294141432,1294141458,0,None,residential,180.464519,0,https://www.openstreetmap.org/#map=19/16.368094/120.629224
1499,529463361,529463375,0,Parana Pine Road,residential,178.057757,0,https://www.openstreetmap.org/#map=19/16.392896/120.589110
821,457422213,1798149690,0,Gladiola Street,residential,134.876939,0,https://www.openstreetmap.org/#map=19/16.387313/120.597775
633,332370309,529462667,0,Manzanillo Road 2,residential,133.543382,0,https://www.openstreetmap.org/#map=19/16.424156/120.589285
2936,1127286712,1127286761,0,Green Lane,residential,125.316213,0,https://www.openstreetmap.org/#map=19/16.417579/120.582951
491,322753164,1253933198,0,Heavens Garden Road,residential,123.506335,0,https://www.openstreetmap.org/#map=19/16.375901/120.611537
541,322760113,537454793,0,Montebello Road 1,residential,122.657469,0,https://www.openstreetmap.org/#map=19/16.381915/120.594400
3305,1294147297,1294147342,0,Pinesville Drive 2,residential,120.814926,0,https://www.openstreetmap.org/#map=19/16.370615/120.628207
1909,531003396,1798230409,0,Alexandria Drive,residential,118.526339,0,https://www.openstreetmap.org/#map=19/16.383806/120.598509


## 6. Unexpected road tags

Tags that are not drivable road classes (such as `bus_stop`) or that hold a list of classes after simplification.

In [6]:
DRIVABLE = {'motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
            'motorway_link', 'trunk_link', 'primary_link', 'secondary_link', 'tertiary_link',
            'living_street', 'service', 'road'}
odd_tags = edges[~edges.highway.isin(DRIVABLE)]
display(odd_tags[['u', 'v', 'key', 'osmid', 'name', 'highway', 'length_m', 'component']])
print('Highway types present:')
print(edges.highway.value_counts())

,u,v,key,osmid,name,highway,length_m,component
5139,4757832782,7908474242,0,"[346317438, 953087702]","['Summer Hill Route', 'Ongasan Road']","['unclassified', 'residential']",844.125604,0
6735,7908474242,4757832782,0,"[953087702, 346317438]","['Summer Hill Route', 'Ongasan Road']","['residential', 'unclassified']",844.125604,0
7519,9386868979,9386868980,0,1017685386,Irisan National High School,bus_stop,21.343297,16
7520,9386868980,9386868979,0,1017685386,Irisan National High School,bus_stop,21.343297,16


Highway types present:
highway
residential                        5824
tertiary                            624
secondary                           598
primary                             366
unclassified                        313
trunk                               298
trunk_link                           31
primary_link                         10
secondary_link                        9
bus_stop                              2
['unclassified', 'residential']       1
['residential', 'unclassified']       1
Name: count, dtype: int64


## 7. Start the correction log

Creates `docs/data_correction_log.csv` with one open row per fragment and per unexpected tag. Fill in `investigation`, `evidence_source`, `action` and `status` as you check each one. The file is only created if it doesn't exist yet, so your notes are never overwritten.

In [7]:
LOG_COLUMNS = ['issue_id', 'dataset', 'feature_id', 'issue', 'investigation',
               'evidence_source', 'action', 'status', 'notes']

if LOG_PATH.exists():
    log = pd.read_csv(LOG_PATH)
    print(f'Log already exists with {len(log)} rows; not overwritten.')
else:
    entries = []
    for r in fragments.itertuples():
        entries.append({
            'issue_id': f'ROAD-{r.component:03d}', 'dataset': 'roads',
            'feature_id': f'component {r.component} ({r.n_nodes} nodes; {r.road_names})',
            'issue': f'Disconnected from main network; {r.dist_to_city_boundary_m} m from city boundary',
            'investigation': '', 'evidence_source': r.osm_link, 'action': '', 'status': 'open', 'notes': '',
        })
    for j, r in enumerate(odd_tags.assign(pair=[frozenset((a, b)) for a, b in zip(odd_tags.u, odd_tags.v)]).drop_duplicates('pair').itertuples(), start=1):
        entries.append({
            'issue_id': f'ROAD-TAG-{j:02d}', 'dataset': 'roads',
            'feature_id': f'osmid {r.osmid} ({r.u}->{r.v})',
            'issue': f'Unexpected highway tag: {r.highway}',
            'investigation': '', 'evidence_source': '', 'action': '', 'status': 'open', 'notes': '',
        })
    log = pd.DataFrame(entries, columns=LOG_COLUMNS)
    log.to_csv(LOG_PATH, index=False)
    print(f'Created {LOG_PATH} with {len(log)} open issues')
display(log)

Log already exists with 50 rows; not overwritten.


,issue_id,dataset,feature_id,issue,investigation,evidence_source,action,status,notes
0,ROAD-001,roads,"component 1 (18 nodes; Almaciga, Cherry, Douglas Fir, Giant Sequoia, Mahogan...",Disconnected from main network; 2246 m from city boundary,Internal roads of Woods Gate Square subdivision. Only connection to main net...,https://www.openstreetmap.org/#map=19/16.390363/120.605253,Exclude from analysis; access restricted by gate.,resolved,NaN
1,ROAD-002,roads,component 2 (14 nodes; (unnamed)),Connects to Poblacion Tuba Road via a service road. The graph excludes highw...,"Fragment (14 nodes, unnamed) connects to Poblacion Tuba Road via a road tagg...",https://www.openstreetmap.org/#map=19/16.394737/120.561598,Team decision 2026-11-24: include public service roads. Graph to be rebuilt ...,resolved,Poblacion Tuba Road may lead outside Baguio (fragment is ~375 m from the cit...
2,ROAD-003,roads,"component 3 (7 nodes; Lias Road, Santo Tomas Central Road)",Disconnected from main network; 416 m from city boundary,"Fragment (Lias Road, Santo Tomas Central Road) connects to Santo Tomas Road ...",https://www.openstreetmap.org/#map=19/16.373391/120.579472; Loganan Road/Sto...,Team decision 2026-11-24: include public service roads. Graph to be rebuilt ...,resolved,Same connecting road as ROAD-004. Loganan Road is a named road; check whethe...
3,ROAD-004,roads,component 4 (7 nodes; Santo Tomas School Area Road),Disconnected from main network; 157 m from city boundary,Fragment (Santo Tomas School Area Road) connects to Santo Tomas Road via Log...,https://www.openstreetmap.org/#map=19/16.372047/120.585778; Loganan Road/Sto...,Team decision 2026-11-24: include public service roads. Graph to be rebuilt ...,resolved,Same connecting road as ROAD-003. Verified: joins the main network in baguio...
4,ROAD-005,roads,component 5 (4 nodes; (unnamed)),Disconnected from main network; 127 m from city boundary,Fragment connects to Santo Tomas Road only through a gated entrance. Satelli...,https://www.openstreetmap.org/#map=19/16.379429/120.566346,Exclude from analysis; access restricted by gate.,resolved,Located about 30 m from ROAD-009 and ROAD-018; likely the same gated area.
5,ROAD-006,roads,component 6 (2 nodes; Woods Gate Square Alley 2),Disconnected from main network; 2453 m from city boundary,Fragment (Woods Gate Square Alley 2) connects to Kennon Road only through a ...,https://www.openstreetmap.org/#map=19/16.389182/120.602779,Exclude from analysis; access restricted by gate.,resolved,Same gated area as ROAD-001.
6,ROAD-007,roads,component 7 (2 nodes; (unnamed)),Disconnected from main network; 1522 m from city boundary,"Fragment (ways 1479800446 and 1466579263, 53 m residential) has dead ends at...",https://www.openstreetmap.org/#map=19/16.381772/120.590733; fragment ways 14...,Exclude from analysis; no drivable connection (footpath only). Not an OSM er...,resolved,Corrected 2026-11-26 after re-check. Still a separate component in baguio_ro...
7,ROAD-008,roads,component 8 (2 nodes; Kellet Street),Disconnected from main network; 2543 m from city boundary,Fragment (Kellet Street) connects to Kennon Road only through the private Wo...,https://www.openstreetmap.org/#map=19/16.390825/120.603794,Exclude from analysis; access restricted by private gate.,resolved,Same gated area as ROAD-001.
8,ROAD-009,roads,component 9 (2 nodes; (unnamed)),Disconnected from main network; 209 m from city boundary,Fragment connects to Santo Tomas Road only through a gated entrance. Satelli...,https://www.openstreetmap.org/#map=19/16.379679/120.567000,Exclude from analysis; access restricted by gate.,resolved,Located about 30 m from ROAD-005; likely the same gated area. Confirm whethe...
9,ROAD-010,roads,component 10 (2 nodes; (unnamed)),Disconnected from main network; 1485 m from city boundary,"Fragment connects to the main network via Santo Nino - Bengao Road, tagged h...",https://www.openstreetmap.org/#map=19/16.387105/120.586544; Bengao Road http...,Team decision 2026-11

In [8]:
log = pd.read_csv(LOG_PATH, dtype=str)
if not log.issue_id.str.startswith('ROAD-STRAIGHT').any():
    rows = []
    for j, r in enumerate(long_straight.itertuples(), start=1):
        c = long_straight.loc[[r.Index]].to_crs(PROJECTED_CRS).geometry.centroid.to_crs(GEOGRAPHIC_CRS).iloc[0]
        rows.append({
            'issue_id': f'ROAD-STRAIGHT-{j:02d}', 'dataset': 'roads',
            'feature_id': f'edge {r.u}->{r.v} (osmid {r.osmid}; {r.name})',
            'issue': f'No stored geometry; drawn as a straight line over {r.length_m:.0f} m',
            'investigation': '',
            'evidence_source': f'{r.osm_link}; https://www.google.com/maps/@{c.y:.6f},{c.x:.6f},19z/data=!3m1!1e3',
            'action': '', 'status': 'open', 'notes': ''})
    log = pd.concat([log, pd.DataFrame(rows)], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    print(f'Added {len(rows)} rows')
display(log[log.issue_id.str.startswith('ROAD-STRAIGHT')][['issue_id', 'feature_id', 'issue']])

,issue_id,feature_id,issue
29,ROAD-STRAIGHT-01,edge 529476141->529463768 (osmid 1368416297; Sunflower Street),No stored geometry; drawn as a straight line over 191 m
30,ROAD-STRAIGHT-02,edge 1294141432->1294141458 (osmid 1124808808; None),No stored geometry; drawn as a straight line over 180 m
31,ROAD-STRAIGHT-03,edge 529463361->529463375 (osmid 497321699; Parana Pine Road),No stored geometry; drawn as a straight line over 178 m
32,ROAD-STRAIGHT-04,edge 457422213->1798149690 (osmid 1316841165; Gladiola Street),No stored geometry; drawn as a straight line over 135 m
33,ROAD-STRAIGHT-05,edge 332370309->529462667 (osmid 42408715; Manzanillo Road 2),No stored geometry; drawn as a straight line over 134 m
34,ROAD-STRAIGHT-06,edge 1127286712->1127286761 (osmid 529121938; Green Lane),No stored geometry; drawn as a straight line over 125 m
35,ROAD-STRAIGHT-07,edge 322753164->1253933198 (osmid 29339048; Heavens Garden Road),No stored geometry; drawn as a straight line over 124 m
36,ROAD-STRAIGHT-08,edge 322760113->537454793 (osmid 42915078; Montebello Road 1),No stored geometry; drawn as a straight line over 123 m
37,ROAD-STRAIGHT-09,edge 1294147297->1294147342 (osmid 114220677; Pinesville Drive 2),No stored geometry; drawn as a straight line over 121 m
38,ROAD-STRAIGHT-10,edge 531003396->1798230409 (osmid 844922520; Alexandria Drive),No stored geometry; drawn as a straight line over 119 m


In [9]:
import re, unicodedata

psa = pd.read_csv(ROOT / 'docs' / 'psa_baguio_barangays.csv', encoding='utf-8-sig')
psa = psa.rename(columns={'10 Digit Code': 'psa_code', 'Barangays': 'psa_name'})
psa['psa_code'] = psa.psa_code.astype(str).str.replace(r'\D', '', regex=True).str.zfill(10)

def norm(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode().lower()
    return re.sub(r'[^a-z0-9]', '', s)

brgy = gpd.read_file(PATHS['barangays'])
brgy['adm4_psgc'] = brgy.adm4_psgc.astype(str).str.zfill(10)
cmp_ = brgy[['adm4_psgc', 'adm4_en']].merge(psa[['psa_code', 'psa_name']],
                                           left_on='adm4_psgc', right_on='psa_code', how='outer')

print(f'File: {len(brgy)} | PSA: {len(psa)}')
print('In file but not in PSA:'); display(cmp_[cmp_.psa_code.isna()][['adm4_psgc', 'adm4_en']])
print('In PSA but not in file:'); display(cmp_[cmp_.adm4_psgc.isna()][['psa_code', 'psa_name']])
matched = cmp_.dropna(subset=['adm4_psgc', 'psa_code'])
mism = matched[matched.adm4_en.map(norm) != matched.psa_name.map(norm)]
print(f'Same code, different name ({len(mism)}):'); display(mism[['adm4_psgc', 'adm4_en', 'psa_name']])

File: 129 | PSA: 129
In file but not in PSA:


,adm4_psgc,adm4_en


In PSA but not in file:


,psa_code,psa_name


Same code, different name (0):


,adm4_psgc,adm4_en,psa_name


In [13]:
import osmnx as ox
from shapely.geometry import LineString, Point
from shapely.ops import substring

BUFFER_M = 500
DRIVE_SERVICE = ox._overpass._get_network_filter('drive_service')
BENGAO = '["highway"="track"]["name"="Santo Nino - Bengao Road"]'   # ROAD-010 team decision
DRIVABLE = {'motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'unclassified', 'residential',
            'motorway_link', 'trunk_link', 'primary_link', 'secondary_link', 'tertiary_link',
            'living_street', 'service', 'road', 'track'}
RANK = ['trunk', 'trunk_link', 'primary', 'primary_link', 'secondary', 'secondary_link',
        'tertiary', 'tertiary_link', 'unclassified', 'residential', 'living_street', 'service', 'road', 'track']

# 1. Download: public roads incl. public service roads, plus the Bengao track road, inside a 500 m buffer
city_proj = gpd.read_file(PATHS['barangays']).to_crs(PROJECTED_CRS).union_all()
boundary = gpd.GeoSeries([city_proj.buffer(BUFFER_M)], crs=PROJECTED_CRS).to_crs(GEOGRAPHIC_CRS).iloc[0]
import time
SERVERS = ['https://maps.mail.ru/osm/tools/overpass/api',
           'https://overpass-api.de/api',
           'https://overpass.kumi.systems/api']
ox.settings.overpass_rate_limit = False
ox.settings.requests_timeout = 300

Gc, wait = None, 30
for attempt in range(4):
    for url in SERVERS:
        ox.settings.overpass_url = url
        try:
            print(f'Trying {url} ...')
            Gc = ox.graph_from_polygon(boundary, custom_filter=[DRIVE_SERVICE, BENGAO], simplify=True,
                                       retain_all=True, truncate_by_edge=True)
            print(f'Success with {url}')
            break
        except Exception as e:
            print(f'  failed: {type(e).__name__}')
    if Gc is not None:
        break
    print(f'All servers busy; waiting {wait} s before trying again')
    time.sleep(wait); wait *= 2
if Gc is None:
    raise RuntimeError('All Overpass servers busy; try again later.')
print('Downloaded:', Gc)
print('Bengao track edges:', sum(d.get('highway') == 'track' for *_, d in Gc.edges(data=True)))

# 2. Remove non-road edges (e.g. bus_stop)
def tags(h): return set(h) if isinstance(h, list) else {h}
bad = [(u, v, k) for u, v, k, d in Gc.edges(keys=True, data=True) if not tags(d.get('highway')) <= DRIVABLE]
Gc.remove_edges_from(bad); Gc.remove_nodes_from(list(nx.isolates(Gc)))
print(f'Removed {len(bad)} non-road edges')

# 3. ROAD-017: join the 0.21 m gap by splitting the Navy Base Road edge at the fragment's end node
FRAG_NODE = 13583131375
def edge_geom(G, u, v, d):
    return d.get('geometry') or LineString([(G.nodes[u]['x'], G.nodes[u]['y']), (G.nodes[v]['x'], G.nodes[v]['y'])])
def metres(geom):
    return gpd.GeoSeries([geom], crs=GEOGRAPHIC_CRS).to_crs(PROJECTED_CRS).iloc[0].length
p = Point(Gc.nodes[FRAG_NODE]['x'], Gc.nodes[FRAG_NODE]['y'])
navy = [(u, v, k, d) for u, v, k, d in Gc.edges(keys=True, data=True)
        if 'Navy Base Road' in str(d.get('name')) and FRAG_NODE not in (u, v)]
U, V = min(((u, v) for u, v, k, d in navy), key=lambda e: edge_geom(Gc, *e, Gc.get_edge_data(*e)[0]).distance(p))
for a, b in [(U, V), (V, U)]:
    for k, d in list((Gc.get_edge_data(a, b) or {}).items()):
        g = edge_geom(Gc, a, b, d); t = g.project(p)
        parts = [LineString(list(substring(g, 0, t).coords[:-1]) + [p.coords[0]]),
                 LineString([p.coords[0]] + list(substring(g, t, g.length).coords[1:]))]
        Gc.remove_edge(a, b, key=k)
        for (s, e), geom in zip([(a, FRAG_NODE), (FRAG_NODE, b)], parts):
            Gc.add_edge(s, e, **dict(d, geometry=geom, length=metres(geom),
                                     correction='ROAD-017: split at fragment end node (0.21 m gap in OSM)'))
Gc.nodes[FRAG_NODE]['street_count'] = len(set(Gc.predecessors(FRAG_NODE)) | set(Gc.successors(FRAG_NODE)))

# 4. One road class per merged edge (ROAD-TAG-01 rule: keep the higher-ranked class)
fixed = 0
for u, v, k, d in Gc.edges(keys=True, data=True):
    if isinstance(d.get('highway'), list):
        d['highway'] = min(d['highway'], key=lambda h: RANK.index(h) if h in RANK else len(RANK)); fixed += 1
    if isinstance(d.get('name'), list):
        d['name'] = ' / '.join(dict.fromkeys(d['name']))
print(f'Single road class assigned to {fixed} merged edges')

# 5. Check the fragments that should now be connected
comps = sorted(nx.weakly_connected_components(Gc), key=len, reverse=True)
main = comps[0]
for issue, node in [('ROAD-010', 7248972404), ('ROAD-017', FRAG_NODE)]:
    print(f'{issue} in main network:', node in main)
print(f'Final graph: {Gc} | components: {len(comps)}')

out_graph = ROOT / 'data' / 'processed' / 'baguio_roads_corrected.graphml'
ox.save_graphml(Gc, out_graph)
print(f'Saved {out_graph}')

Trying https://maps.mail.ru/osm/tools/overpass/api ...
Success with https://maps.mail.ru/osm/tools/overpass/api
Downloaded: MultiDiGraph with 6714 nodes and 14247 edges
Bengao track edges: 2
Removed 2 non-road edges
Single road class assigned to 29 merged edges
ROAD-010 in main network: True
ROAD-017 in main network: True
Final graph: MultiDiGraph with 6712 nodes and 14247 edges | components: 33
Saved /Users/francie/Downloads/baguio-road-accessibility-ml/data/processed/baguio_roads_corrected.graphml


In [14]:
orig = nx.read_graphml(RAW / 'baguio_roads_full.graphml')
corr = nx.read_graphml(ROOT / 'data' / 'processed' / 'baguio_roads_corrected.graphml')
def stats(g):
    comps = sorted(nx.weakly_connected_components(g), key=len, reverse=True)
    hw = pd.Series([str(d.get('highway')) for *_, d in g.edges(data=True)]).value_counts()
    return {'nodes': g.number_of_nodes(), 'edges': g.number_of_edges(), 'components': len(comps),
            'main network nodes': len(comps[0]), 'service edges': hw.get('service', 0),
            'track edges': hw.get('track', 0), 'bus_stop edges': hw.get('bus_stop', 0),
            'list road classes': int(hw[hw.index.str.startswith('[')].sum()),
            'corrected edges (ROAD-017)': sum('correction' in d for *_, d in g.edges(data=True))}
display(pd.DataFrame({'original': stats(orig), 'corrected': stats(corr)}))

,original,corrected
nodes,3745,6712
edges,8077,14247
components,28,33
main network nodes,3656,6592
service edges,0,3048
track edges,0,2
bus_stop edges,2,0
list road classes,2,0
corrected edges (ROAD-017),0,4
